<a href="https://colab.research.google.com/github/WallofWonder/Hugging-Face-RL-Learning/blob/main/notebooks/unit3/unit3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 单元 3：使用 RL Baselines3 Zoo 玩雅达利游戏 (Atari) 👾 进阶深层 Q 网络 (Deep Q-Learning)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/thumbnail.jpg" alt="Unit 3 Thumbnail">

在本 Notebook 中，**你将训练一个玩《太空侵略者》(Space Invaders) 的 Deep Q-Learning 智能体**。我们将使用 [RL Baselines3 Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)，这是一个基于 [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) 的训练框架，提供了用于训练、评估智能体、调整超参数、绘制结果和录制视频的脚本。

我们使用的是 [RL-Baselines-3 Zoo 集成的原生版本 Deep Q-Learning](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html)，不包含 Double-DQN、Dueling-DQN 和优先经验回放 (Prioritized Experience Replay) 等扩展。

⬇️ 下面是你**即将实现的目标**示例 ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/ThomasSimonini/ppo-SpaceInvadersNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

### 🎮 运行环境：

- [SpacesInvadersNoFrameskip-v4](https://gymnasium.farama.org/environments/atari/space_invaders/)

你可以在这里查看不同版本《太空侵略者》之间的区别 👉 https://gymnasium.farama.org/environments/atari/space_invaders/#variants

### 📚 强化学习库：

- [RL-Baselines3-Zoo](https://github.com/DLR-RM/rl-baselines3-zoo)

## 本 Notebook 的目标 🏆
在学习完本 Notebook 后，你将能够：
- 深入理解 **RL Baselines3 Zoo 的工作原理**。
- 能够**将你训练好的智能体和代码推送到 Hugging Face Hub**，并附带精彩的视频回放和评估分数 🔥。

## 本教程来自 深度强化学习课程 (Deep Reinforcement Learning Course)
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中，你将：

- 📖 系统地在**理论与实践**中学习深度强化学习。
- 🧑‍💻 学习**使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的环境中训练智能体**

查看更多课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">订阅课程</a>**（我们会收集您的邮箱，以便在每个单元发布时向您发送链接，并提供有关挑战和更新的信息）。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区和我们进行交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前置准备 🏗️
在开始本 Notebook 之前，你需要：

🔲 📚 **[通过阅读第 3 单元学习 Deep Q-Learning](https://huggingface.co/deep-rl-course/unit3/introduction)** 🤗

我们一直在努力改进我们的教程，因此**如果你在此 Notebook 中发现任何问题**，请在 [Github 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

# 让我们训练一个玩《太空侵略者》 👾 的 Deep Q-Learning 智能体并将其上传到 Hub。

我们强烈建议学生**使用 Google Colab 进行动手练习，而不是在个人电脑上运行它们**。

通过使用 Google Colab，**你可以专注于学习和实验，而无需担心设置环境的技术细节**。

为了通过认证流程，你需要将训练好的模型推送到 Hub，并**获得 >= 200 的分数**。

要查看你的结果，请转到排行榜并找到你的模型，**结果计算公式：result = mean_reward - std of reward**

有关认证流程的更多信息，请查看此部分 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 温馨提示 💡
最好在你的 Google 云端硬盘 (Google Drive) 中保存一份此 Colab 的副本，这样**如果发生超时断开**，你仍然可以将保存的 Notebook 保留在云端硬盘中，而不必从头开始填写所有内容。

你可以通过按 `Ctrl + S` 或点击 `文件 > 在云端硬盘中保存副本` 来实现。

此外，我们将**使用 1M (100万) 的时间步长训练大约 90 分钟**。输入 `!nvidia-smi` 可以查看你正在使用什么型号的 GPU。

如果你想训练更多步数（如 1000 万步），这大约需要 9 个小时，很可能会导致 Colab 超时。在这种情况下，我建议在本地计算机（或其他地方）运行。只需点击 `文件 > 下载` 即可。

## 设置 GPU 💪
- 为了**加速智能体的训练，我们将使用 GPU**。为此，请转到 `代码执行程序 > 更改运行时类型`。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- 选择 `硬件加速器 > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

# 安装 RL-Baselines3 Zoo 及其依赖项 📚

如果你看到 `ERROR: pip's dependency resolver does not currently take into account all the packages that are installed.` **这是正常现象，并非关键错误**，这只是版本冲突，我们需要的包已经成功安装了。

In [ ]:
!pip install git+https://github.com/DLR-RM/rl-baselines3-zoo

In [ ]:
!apt-get install swig cmake ffmpeg

为了能够在 Gymnasium 中使用雅达利游戏，我们需要安装 atari 包，并使用 `accept-rom-license` 来下载 ROM 文件（游戏文件）。

In [ ]:
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]

## 创建虚拟显示器 🔽

在此 Notebook 的运行过程中，我们需要生成回放视频。为此，在 Colab 中，**我们需要一个虚拟屏幕以便能够渲染环境**（从而记录帧）。

因此，以下单元将安装相应的库并创建并运行一个虚拟屏幕 🖥。

In [ ]:
%%capture
!apt install python-opengl
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 训练我们的 Deep Q-Learning 智能体玩《太空侵略者》 👾

要使用 RL-Baselines3-Zoo 训练智能体，我们只需要做两件事：

1. 创建一个名为 `dqn.yml` 的超参数配置文件，其中将包含我们的训练超参数。

这是一个模板示例：

```
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: 'CnnPolicy'
  n_timesteps: !!float 1e6
  buffer_size: 100000
  learning_rate: !!float 1e-4
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  # 如果为 True，您需要停用 replay_buffer_kwargs 中的 handle_timeout_termination
  optimize_memory_usage: False
```

在这里我们可以看到：
- 我们使用 `Atari Wrapper` 预处理输入（帧缩减、灰度化、堆叠 4 帧）
- 我们使用 `CnnPolicy`，因为我们使用卷积层来处理游戏帧图像
- 我们训练 1000 万个 `n_timesteps`
- 经验回放缓冲区 (Memory Buffer) 的大小为 100000，即你保存并用来再次训练智能体的经验步数。

💡 我的建议是**将训练步数减少到 1M (100万步)**，在 P100 GPU 上大约需要 90 分钟。输入 `!nvidia-smi` 会告诉你当前使用的是哪种 GPU。如果训练 1000 万步，大约需要 9 个小时，这很可能会导致 Colab 超时。我建议在你的本地电脑（或其他地方）上运行。只需点击：`文件 > 下载` 即可。

在超参数优化方面，我建议重点关注以下 3 个超参数：
- `learning_rate` (学习率)
- `buffer_size` (经验回放缓冲区大小)
- `batch_size` (批大小)

作为良好的实践，你需要**阅读文档以了解每个超参数的具体作用**：https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html#parameters

2. 我们开始训练并将模型保存到 `logs` 文件夹 📁 中

- 在 `--algo` 后指定算法，在 `-f` 后指定保存模型的目录，并在 `-c` 后指定超参数配置文件的路径。

In [ ]:
!python -m rl_zoo3.train --algo ________ --env SpaceInvadersNoFrameskip-v4  -f _________  -c _________

#### 解决方案

In [ ]:
!python -m rl_zoo3.train --algo dqn  --env SpaceInvadersNoFrameskip-v4 -f logs/ -c dqn.yml

## 让我们评估一下我们的智能体 👀
- RL-Baselines3-Zoo 提供了 `enjoy.py`，这是一个用于评估我们智能体的 Python 脚本。在大多数 RL 库中，我们都将评估脚本称为 `enjoy.py`。
- 让我们对其进行 5000 个时间步长的评估 🔥

In [ ]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps _________  --folder logs/

#### 解决方案

In [ ]:
!python -m rl_zoo3.enjoy  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --no-render  --n-timesteps 5000  --folder logs/

## 在 Hub 上发布我们训练好的模型 🚀
既然我们在训练后看到了不错的结果，我们就可以用一行代码将我们训练好的智能体发布到 Hub 🤗 上。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/space-invaders-model.gif" alt="Space Invaders model">

通过使用 `rl_zoo3.push_to_hub`，**你将对智能体进行评估、录制回放、生成模型卡片并将其推送到 Hub**。

这样：
- 你可以**展示你的工作成果** 🔥
- 你可以**直观地看到你的智能体在玩游戏** 👀
- 你可以**与社区分享一个其他人也可以直接使用的智能体** 💾
- 你可以**访问排行榜 🏆 来查看你的智能体与同学相比的表现** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard

为了能够与社区分享你的模型，还有另外三个步骤需要遵循：

1️⃣ （如果尚未注册）在 HF 上创建一个账户 ➡ https://huggingface.co/join

2️⃣ 登录，然后你需要存储来自 Hugging Face 网站的身份验证 Token。
- 创建一个具有 **write (写入) 权限** 的新 Token (https://huggingface.co/settings/tokens)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制该 Token
- 运行下面的单元格并粘贴该 Token

In [ ]:
from huggingface_hub import notebook_login # To log to our Hugging Face account to be able to upload models to the Hub.
notebook_login()
!git config --global credential.helper store

如果你不想使用 Google Colab 或 Jupyter Notebook，则需要改用此命令：`huggingface-cli login`

3️⃣ 我们现在准备好将我们训练有素的智能体推送到 🤗 Hub 🔥

让我们运行 `push_to_hub.py` 文件将我们训练好的智能体上传到 Hub。

`--repo-name`：仓库的名称

`-orga`：你的 Hugging Face 用户名

`-f`：训练好的模型文件夹所在的位置（在我们的例子中是 `logs`）

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit3/select-id.png" alt="Select Id">

In [ ]:
!python -m rl_zoo3.push_to_hub  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --repo-name _____________________ -orga _____________________ -f logs/

#### 解决方案

In [ ]:
!python -m rl_zoo3.push_to_hub  --algo dqn  --env SpaceInvadersNoFrameskip-v4  --repo-name dqn-SpaceInvadersNoFrameskip-v4  -orga ThomasSimonini  -f logs/

### .

恭喜 🥳 你刚刚使用 RL-Baselines-3 Zoo 训练并上传了你的第一个 Deep Q-Learning 智能体。上面的脚本应该已经显示了一个模型库的链接，例如 https://huggingface.co/ThomasSimonini/dqn-SpaceInvadersNoFrameskip-v4。当你访问此链接时，你可以：

- 在右侧看到**智能体的视频预览**。
- 点击 "Files and versions" (文件和版本) 查看仓库中的所有文件。
- 点击 "Use in stable-baselines3" (在 stable-baselines3 中使用) 获取显示如何加载模型的代码片段。
- 一个模型卡片（`README.md` 文件），它给出了模型的描述以及你使用的超参数。

在底层，Hub 使用基于 git 的存储库（如果你不知道 git 是什么，不用担心），这意味着你可以随着实验和改进智能体，用新版本更新模型。

使用 [排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 🏆 **将你的智能体结果与你的同学进行比较**

## 加载一个强大的已训练模型 🔥
- Stable-Baselines3 团队在 Hub 上上传了 **150 多个训练有素的深度强化学习智能体**。

你可以在这里找到它们：👉 https://huggingface.co/sb3

一些例子：
- 🚀 经典 Asteroids: https://huggingface.co/sb3/dqn-AsteroidsNoFrameskip-v4
- 🛸 Beam Rider: https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4
- 🧱 Breakout: https://huggingface.co/sb3/dqn-BreakoutNoFrameskip-v4
- 🏃‍♂️ Road Runner: https://huggingface.co/sb3/dqn-RoadRunnerNoFrameskip-v4

让我们加载一个玩 Beam Rider 的智能体：https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4/resolve/main/replay.mp4" type="video/mp4"></video>

1. 我们使用 `rl_zoo3.load_from_hub` 下载模型，并将其放入一个可以称为 `rl_trained` 的新文件夹中

In [ ]:
# Download model and save it into the logs/ folder
!python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/

2. 让我们对它进行 5000 个时间步的评估

In [ ]:
!python -m rl_zoo3.enjoy --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000  -f rl_trained/ --no-render

为什么不尝试训练你自己的 **Deep Q-Learning 智能体玩 BeamRiderNoFrameskip-v4 呢？🏆**

如果你想尝试，请查看 https://huggingface.co/sb3/dqn-BeamRiderNoFrameskip-v4#hyperparameters **在模型卡片中，有已训练智能体的超参数。**

但是寻找超参数可能是一项艰巨的任务。幸运的是，我们将在下一个单元中看到如何**使用 Optuna 优化超参数 🔥。**

## 一些额外的挑战 🏆
最好的学习方法**就是自己动手尝试**！

在 [排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 中你会找到你的智能体。你能名列前茅吗？

这里是可以尝试用来训练智能体的环境列表：
- BeamRiderNoFrameskip-v4
- BreakoutNoFrameskip-v4
- EnduroNoFrameskip-v4
- PongNoFrameskip-v4

此外，**如果你想自己学习实现 Deep Q-Learning**，你绝对应该看看 CleanRL 的实现：https://github.com/vwxyzjn/cleanrl/blob/master/cleanrl/dqn_atari.py

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>

________________________________________________________________________
恭喜你完成本章！

如果你对所有这些元素仍然感到困惑……这完全正常！**对于我和所有学习强化学习的人来说都是一样的。**

在继续并尝试额外挑战之前，花点时间**真正掌握这些材料**。掌握这些元素并打下坚实的基础非常重要。

在下一个单元中，**我们将学习 [Optuna](https://optuna.org/)**。深度强化学习中最关键的任务之一是找到一组优秀的训练超参数。而 Optuna 是一个可以帮助您自动进行搜索的库。

### 这是一个为您而建的课程 👷🏿‍♀️

最后，我们希望根据您的反馈不断改进和更新课程。如果您有一些建议，请填写此表单 👉 https://forms.gle/3HgA7bEHwAmmLfwh9

我们一直在努力改进我们的教程，因此**如果您在此 Notebook 中发现任何问题**，请在 [Github 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

期待在 Bonus unit 2 与你相见！🔥

### 保持学习，保持优秀 🤗